In [ ]:
import os
from azure.search.documents.indexes import SearchIndexClient
from azure.identity import DefaultAzureCredential, AzureCliCredential, InteractiveBrowserCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    FabricDataAgentKnowledgeSource,
    FabricDataAgentKnowledgeSourceParameters,
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import KnowledgeRetrievalLowReasoningEffort, KnowledgeRetrievalMediumReasoningEffort

from dotenv import load_dotenv

load_dotenv(override=True)

aoai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
aoai_gpt_deployment = os.getenv("AZURE_OPENAI_GPT_DEPLOYMENT")
aoai_gpt_model = os.getenv("AZURE_OPENAI_GPT_MODEL")

tenant_id = os.getenv("AZURE_TENANT_ID")
credential = DefaultAzureCredential() if not tenant_id else AzureCliCredential(tenant_id=tenant_id)

search_endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")

In [ ]:
# List knowledge sources by name and type
index_client = SearchIndexClient(endpoint = search_endpoint, credential = credential)

for ks in index_client.list_knowledge_sources():
    print(f"  - {ks.name} ({ks.kind})")


In [ ]:
# Create a knowledge source
knowledge_source_name = "ks-fabric-dataagent01"

index_client = SearchIndexClient(
    endpoint=search_endpoint,
    credential=credential
)

knowledge_source = FabricDataAgentKnowledgeSource(
    name=knowledge_source_name,
    description="Order database including Order Details.",
    fabric_data_agent_parameters=FabricDataAgentKnowledgeSourceParameters(
        workspace_id=os.getenv("FABRIC_WORKSPACE_ID"),
        data_agent_id=os.getenv("FABRIC_DATA_AGENT_ID")
    )
)

index_client.create_or_update_knowledge_source(knowledge_source)

In [ ]:
# List knowledge bases by name
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient

for kb in index_client.list_knowledge_bases():
    print(f"  - {kb.name}")

In [ ]:
# Create a knowledge base
aoai_params = AzureOpenAIVectorizerParameters(
    resource_url = aoai_endpoint,
    deployment_name = aoai_gpt_deployment,
    model_name = aoai_gpt_model,
)

knowledge_base = KnowledgeBase(
    name = "kb-fabric-dataagent01",
    description = "This knowledge base handles queries related to order database searches, including order details.",
    retrieval_instructions = "Use ks-fabric-dataagent01 to search the Order database and retrieve order details using the order user's name.",
    answer_instructions = "Use the retrieved order details to provide accurate responses and include the associated product ID.",
    output_mode = "answerSynthesis",
    knowledge_sources = [
        KnowledgeSourceReference(name = "ks-fabric-dataagent01"),
    ],
    models = [KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters = aoai_params)],
    encryption_key = None,
    retrieval_reasoning_effort = KnowledgeRetrievalMediumReasoningEffort(),
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{knowledge_base.name}' created or updated successfully.")